# NB03: Data Analysis

**LSE ME204 – Data Engineering Principles for the Social Sciences (2026)**

**LSE ID:** 250093214

## Setup

Run the cell below to ensure all required packages are installed before running all other cells.

In [1]:
import json
import requests
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

Now it is time to take the data tables and transform them for analysis.

## Methodology

### How will Pokemon with changes in BSTs between Generations be handled?

This analysis does not account for Pokemon whose BSTs have changed between generations. Ideally, a Pokemon's original BST would be used in all analyses, and the API does provide information about the original stats of Pokemon that have had theirs changed. However, the API does not provide the full original stat spread, only the original value that had been altered. Due to personal restrictions, I elected to simply use the changed stat spreads. The later-generation changes in BSTs are small enough that I do not believe they will introduce a massive skew in the data, but future analyses can take these altered BSTs into account.

### What is the baseline that power creep will be measured against?

Generation 1 is going to be the baseline against which power creep will be measured, since power creep is about the strengthening of the pieces of a game over time. If power creep has not occurred, then the average BSTs between Generations 1 and 9 should be about the same, with minor fluctuations between Generations. If power creep has occurred, however, there should be a marked difference between the average BSTs between Generations 1 and 9, with average BST slowly increasing over time.

It is possible to set the baseline against which power creep is measured elsewhere, such as the 2D vs. 3D Generations of games. This would set Generations 1-5 as the baseline against which Generations 6-9 are compared. I chose to set it at Generation 1 because this creates more data points for comparison and creates a larger timeline over which power creep can be considered.

## The desired graphs

I do not know entirely what I want to find in this data at the moment, but I do have some idea about what graphs I want to create that will hopefully reveal trends in the data. I want to create line graphs of how the average BST of Generations have changed as a whole and how the average BST of Standard, Legendary, and Mythical Pokemon have changed. This will reveal trends over time, hence the use of a line graph, and provide the clearest example of power creep in graphical form. I also want to separate Pokemon by category to see if the change is BST is more pronounced in one category over another.

I also know that I would like to create histograms showing the distribution of all Pokemons' BSTs, with the different bars colored by generation. Once again, there are going to be different graphs where all Pokemon are included, and where the Pokemon are separated by Standard, Legendary, and Mythical classifications. I think a histogram would be the best to show the distribution since there are a lot of Pokemon with similar BSTs, so using a scatterplot might hide some data points. Boxplots might also run into the same issue.

I have been focusing a lot on BSTs, but if I do find evidence of strong power creep, I would like to see if there is one particular stat that has been increasing a lot over Generations, or if all of them have been increasing equally. I would like to create line graphs faceted by stat showing the change in the average base stats over each Generation, and then another set of line graphs where the Pokemon are divided by their Classifications.

Similarly, I would like to create histograms showing the distribution of all of the base stats for all Pokemon, faceted by base stat. This is going to require a lot of graphs, as I want to do one for all Pokemon, as well as sets of histograms for each category of Pokemon. A histogram was chosen because I fear that a scatterplot or a box plot would hide some of the data points.

## Reading the CSVs

In [2]:
df_stats = pd.read_csv('../data/processed/stats_table.csv')

df_base = pd.read_csv('../data/processed/base_table.csv')

## Creating the graphs

### Line graph of average BSTs over Generations, both aggregated and separated by classification

In [4]:
dfg = (
    df_base
    .groupby(['gen_num'], as_index = False)
    ['BST'].mean()
)

dfg

,gen_num,BST
0,Gen 1,407.642384
1,Gen 2,407.180000
2,Gen 3,403.725926
3,Gen 4,445.570093
4,Gen 5,425.756410
5,Gen 6,429.305556
6,Gen 7,449.409091
7,Gen 8,439.218750
8,Gen 9,457.391667


In [28]:
avg_bst_lg = px.line(
    dfg,
    x = 'gen_num',
    y = 'BST',
    title = 'Over generations, average BSTs of all Pokemon have increased since Generation 4, though not at a consistent rate',
    subtitle = 'Average base stat totals (BSTs) over Generations',
    labels = {
        'gen_num':'Generation',
        'BST':'Average Base Stat Totals'
    },
    markers = True
)

avg_bst_lg.update_yaxes(rangemode='tozero')

avg_bst_lg.show()

In [6]:
dfgc = (
    df_base
    .groupby(['classification','gen_num'], as_index = False)
    ['BST'].mean()
)

dfgc

,classification,gen_num,BST
0,legendary,Gen 1,605.000000
1,legendary,Gen 2,620.000000
2,legendary,Gen 3,620.000000
3,legendary,Gen 4,625.555556
4,legendary,Gen 5,613.333333
5,legendary,Gen 6,653.333333
6,legendary,Gen 7,540.363636
7,legendary,Gen 8,576.818182
8,legendary,Gen 9,571.363636
9,mythical,Gen 1,600.000000


In [7]:
class_bst_lg = px.line(
    dfgc,
    x = 'gen_num',
    y = 'BST',
    color = 'classification',
    title = 'Over generations, Legendary and Mythical Pokemon have stayed at relatively the same power level, while Standard Pokemon have increased at an inconsistent rate',
    subtitle = 'Average Pokemon Base Stat Totals (BSTs) by Generation separated by Pokemon Classification',
    labels = {
        'gen_num':'Generation',
        'BST':'Average Base Stat Totals'
    },
    markers = True,
    height = 1000,
    width = 1500
)

class_bst_lg.update_yaxes(rangemode='tozero')

class_bst_lg.show()

From these graphs, I can see that over time, Pokemons' BSTs have been increasing, at least when it comes to Standard Pokemon. However, this increase has not been consistent (the line graph is not a straight, upward slope). This increase also does not hold for all types of Pokemon; Legendary and Mythical Pokemon have stayed relatively consistent over Generations, until Generation 7 when their power took a large dip. Mythical power levels returned to where they were prior, around the 600 BST mark, while Legendaries did not recover. I would wager, however, that this decrease is due to my analysis using weaker versions of later-generation Legendaries rather than their fully-powered forms. The Generation 7 Legendary dip, I would wager, is from having a Legendary evolutionary line where the earlier Pokemon in the line are weaker than their fully-evolved counterparts. Still, these graphs provide evidence of at least some power creep occurring in Pokemon.

### Histograms showing the distribution of BSTs, both aggregated and separated by classification

Note: much of this histogram code comes from the [plotly documentation on building histograms](https://plotly.com/python/histograms/#styled-histogram).

In [8]:
def create_histogram(dataframe, filter, filter_list, desired_data, x_start, x_end, x_size,color_list):
##Builds a histogram given a dataframe, the desired data, the size of the bins, a color list##
    hist = go.Figure() ##Creates a graph object
    for i in range(len(filter_list)):
        df_gen = dataframe[dataframe[filter] == filter_list[i]]
        hist.add_trace(go.Histogram( ## Adds a new facet to the figure, which in this case is a histogram
            x = df_gen[desired_data],
            name = filter_list[i], ## What label will show when you mouse over a bar
            xbins = { ## Determines the size of the bins on the x-axis
                'start':x_start,
                'end':x_end,
                'size':x_size
            },
            marker_color = color_list[i],
        )
        )
    return hist


In [9]:
gen_list = ['Gen 1', 'Gen 2', 'Gen 3', 'Gen 4', 'Gen 5', 'Gen 6', 'Gen 7', 'Gen 8', 'Gen 9']
color_list = ['#EE4B2B', '#EFBF04', '#00674F', '#B9F2FF', '#000000', '#305CDE', '#654321', '#898989', '#7F00FF']

In [10]:
bst_hist = create_histogram(df_base, 'gen_num', gen_list, 'BST', 0, 1000, 50, color_list)

bst_hist.update_layout(
    xaxis_title_text='Base Stat Total (BST)', 
    yaxis_title_text='Count of Pokemon', 
    bargap=0.2, # According to the documentation, adds a gap between bars of different bin sizes (ex: between the 400-449 and the 450-499 bins)
    bargroupgap=0.2 # According to the documentation, adds a gap between bars of the same bin sizes (ex: all the bars in the 400-449 bin)
)

bst_hist.update_layout(
    title = go.layout.Title( ##Lets me change the title and the font of the title
        text = 'Over time, higher BST Pokemon have become more common while lower BST Pokemon have become less common.',
        font=dict(size=11)
    )
)

bst_hist.show()

In [11]:
def create_histogram_class(dataframe, filter, filter_list, filter_two, filter_two_type, desired_data, x_start, x_end, x_size,color_list):
##Builds a histogram given a dataframe, the desired data, the size of the bins, a color list. Adds a second filter for further data slicing.##
    hist = go.Figure() ##Creates a graph object
    for i in range(len(filter_list)):
        df_gen = dataframe[(dataframe[filter] == filter_list[i]) & (dataframe[filter_two] == filter_two_type)]
        hist.add_trace(go.Histogram( ## Adds a new facet to the figure, which in this case is a histogram
            x = df_gen[desired_data],
            name = filter_list[i], ## What label will show when you mouse over a bar
            xbins = { ## Determines the size of the bins on the x-axis
                'start':x_start,
                'end':x_end,
                'size':x_size
            },
            marker_color = color_list[i],
        )
        )
    return hist

In [12]:
bst_hist_std = create_histogram_class(df_base, 'gen_num', gen_list, 'classification', 'standard', 'BST', 0, 1000, 50, color_list)

bst_hist_std.update_layout(
    xaxis_title_text='Base stat total',
    yaxis_title_text='Count of Standard Pokemon',
    bargap=0.2,
    bargroupgap=0.2
)

bst_hist_std.update_layout(
    title = go.layout.Title(
        text = 'Standard Pokemon, overall, have become increasingly powerful over generations, based on the amount of Pokemon in higher BST bins.',
        font=dict(size=16)
    )
)

bst_hist_std.show()

In [13]:
bst_hist_leg = create_histogram_class(df_base, 'gen_num', gen_list, 'classification', 'legendary', 'BST', 0, 1000, 50, color_list)

bst_hist_leg.update_layout(
    xaxis_title_text='Base stat total',
    yaxis_title_text='Count of Legendary Pokemon',
    bargap=0.2,
    bargroupgap=0.2
)

bst_hist_leg.update_layout(
    title = go.layout.Title(
        text = 'Legendary Pokemon have stayed at about the same power level over Generations, but more Legendary Pokemon have been added in newer Generations than older',
        font=dict(size=13)
    )
)

bst_hist_leg.show()

In [14]:
bst_hist_myth = create_histogram_class(df_base, 'gen_num', gen_list, 'classification', 'mythical', 'BST', 0, 1000, 50, color_list)

bst_hist_myth.update_layout(
    xaxis_title_text='Base stat total',
    yaxis_title_text='Count of Mythical Pokemon',
    bargap=0.2,
    bargroupgap=0.2
)

bst_hist_myth.update_layout(
    title = go.layout.Title(
        text = 'Mythical Pokemon have stayed at about the same power level over Generations, but more Mythicals per Generation were added from Generations 3 - 7.',
        font=dict(size=13)
    )
)

bst_hist_myth.show()

These histograms were a good starting point for my investigations into the distribution of BSTs, especially for Legendaries and Mythicals. However, this is likely because Legendaries and Mythicals have less data points, so their histograms were less crowded overall. I was able to glean some insight from the Standard Pokemon and all Pokemon histograms, but with how messy the histograms are, it does not make for the best visual to make an argument with. As such, it's time to go back to the drawing board and look for other ways to visualize the distribution of BSTs for use in data storytelling. Still, more powerful Pokemon are becoming more common.

Also, using plotly Graph Objects rather than plotly Express makes creating graphs the way I want to harder, as I cannot set a subtitle, nor change the height and width of my graph, so I will be going back to plotly Express for future plots. 

### Alternate ways of visualizing the spread of BSTs

#### Faceted histogram

In [15]:
std_df = df_base[df_base['classification'] == 'standard']
leg_df = df_base[df_base['classification'] == 'legendary']
myth_df = df_base[df_base['classification'] == 'mythical']

In [16]:
bst_hist = px.histogram(
    std_df,
    x = 'BST',
    color = 'gen_num',
    facet_row = 'gen_num',
    width = 700,
    height = 2000
)

bst_hist.show()

The faceted histogram is easier to read than the colored histograms made earlier, but I find it doesn't really say the same things as the earlier histogram did. Still, this was just a test for potentially displaying BST data for an eventual report. I will return to the idea later if I do not find something that works better.

#### Boxplots

In [17]:
def create_boxplot(dataframe, x_plot, y_plot, title_text, subtitle_text, wdth):
##Creates a box plot given a dataframe, the data to plot on the x and y axes, title and subtitle text, and a width of the graph.##
    boxplot = px.box(
        dataframe,
        x = x_plot,
        y = y_plot,
        title = title_text,
        subtitle = subtitle_text,
        width = wdth
    )

    return boxplot

In [18]:
base_box = create_boxplot(df_base, 'gen_num', 'BST', 'The lower quartile of Pokemon BST has not shifted over generations, but since Generation 5, the upper quartile has been steadily pushed over 500 BST', 'Additionally, since Gen 4, the median of Pokemon BST has been steadily going upward. Distribution of Pokemon base stat totals', 2000)

base_box.show()

In [19]:
base_std_box = create_boxplot(std_df, 'gen_num', 'BST', 'The median Standard Pokemon BST has been higher than Genenartion 1 since Generation 4, with the largest increases starting in Generation 7', 'Distrtibution of Standard Pokemon base stat totals', 2000)

base_std_box.show()

In [20]:
base_leg_box = create_boxplot(leg_df, 'gen_num','BST', 'Legendaries have stayed around generally the same power level over Generations', 'The outliers are likely due to the introduction of legendary evolutionary lines, with pre-evos of Legendaries being weaker than their fully-evolved counterparts (ex: Cosmog and Cosmoem being pre-evolutions of Solgaleo/Lunala). Distribution of Legendary base stat totals', 2000)

base_leg_box.show()

In [21]:
base_myth_box = create_boxplot(myth_df, 'gen_num','BST', 'Mythicals have stayed around generally the same power level over Generations', 'Outliers are once again likely due to pre-evolutions of a mythical being weaker than its final forms. Distribution of Mythical base stat totals', 2000)

base_myth_box.show()

I mentioned earlier in the notebook that I believed boxplots would hide some of the datapoints, and looking at the boxplots side-by-side with the histograms, I was right. Though the boxplots show the distribution for all Pokemon and Standard Pokemon in a more readable way, it is less obvious that there are less Legendaries and Mythicals overall. The boxplots also do not show the increasing number of Legendaries and Mythicals added with each generation.

#### Scatterplots

In [22]:
def create_scatter(dataframe, x_plot, y_plot, color_map, title_text, subtitle_text, wdth):
##Creates a scatterplot given a dataframe, the data to plot on the x and y axes, title and subtitle text, and the width of the graph##
    scatterplot = px.scatter(
        dataframe,
        x = x_plot,
        y = y_plot,
        color = color_map,
        title = title_text,
        subtitle = subtitle_text,
        width = wdth
    )

    return scatterplot

In [23]:
base_scatter = create_scatter(df_base, 'BST', 'gen_num', 'gen_num', 'A similar number of Standard Pokemon are over the 500 BST threshold in each Generation, despite the overall number of Pokemon in each Generation generally decreasing', 'Base stat totals of all Pokemon across Generations', 2000)

base_scatter.show()

In [24]:
base_scatter_std = create_scatter(std_df, 'BST', 'gen_num', 'gen_num', 'A similar number of Standard Pokemon are over the 500 BST threshold in each Generation, despite the overall number of Pokemon in each Generation generally decreasing', 'Base stat totals of Standard Pokemon across Generations', 2000)

base_scatter_std.show()

In [ ]:
base_scatter_leg = create_scatter(leg_df, 'BST', 'gen_num', 'gen_num', 'More legendaries are being added every Generation, though they generally stay around the same power level', 'Base stat totals for Legendary Pokemon across Generations', 2000)

base_scatter_leg.show()

In [25]:
base_scatter_myth = create_scatter(myth_df, 'BST', 'gen_num', 'gen_num', 'There are more mythicals in later Generations than in earlier Generations', 'Base stat totals of Mythical Pokemon across Generations', 2000)

base_scatter_myth.show()

The scatterplots do a good job of showcasing the data, but I just do not like how they look. Once again, this type of visualization does not work for all classifications. The Legendaries and Mythicals scatterplot shows all of the data points clearly, but the graphs look too sparse. The Standard scatterplot and the combined classifications scatterplot's data points are too clustered together, which makes it hard to read the data clearly. Once again, not a bad manner of visualization, but I think something else could work better.

Overall, I think the faceted histograms did the best job displaying the data.

## Line graph of average base stats across Generations, both aggregated and separated by classification

In order to separate out the average base stats by Generation, I'm going to actually need to add a Generation column on the dataframe where the base stats data is. I could do this in one of two ways: I could create a dictionary with each Generation as the key and a list of all of the Pokemon added in said Generation as the value. Then, I would build a funcion that goes through each Generation key and checks to see if the Pokemon name is in the list of Pokemon added in that Generation. Depending on what Generation the Pokemon was added in, that Generation would then be assigned to that row. I could even use the .notna() function to perform the same filtering of Pokemon that I did on the species dataframe that contains the BST data.

The second way I could do this is just by merging the species and stats dataframes on the name key, which would automatically assign the generation to the right Pokemon and filter out the Pokemon I don't want to include in my analysis because there would be no match in the 'pokemon_name' column to adhere the rows to. This would also require a lot less code.

I am going to merge the two dataframes. Originally I did this process in NB02, but in the interest of maintaining clean tables for analysis, I am moving the merge here. Maybe there will be a use in the future for the raw stats table that I cannot yet see, and pre-merging the data would take away some of the analysis that could be done on the table. I will know the merge is successful if I have 6150 rows (6 stats * 1025 Pokemon in the analysis)

Before I perform the merge, I am going to have to rename the 'name' column to 'pokemon_name' so there will actually be a common column to merge on.

In [30]:
df_stats = df_stats.rename(columns = {'name':'pokemon_name'})

In [35]:
merge_df = pd.merge(df_base,df_stats,how='left',on='pokemon_name').drop(columns = 'id_y').rename(columns = {'id_x':'id'})

In [36]:
merge_df

,pokemon_name,id,BST,gen_num,classification,base_stat,stat_name
0,bulbasaur,1,318,Gen 1,standard,45,hp
1,bulbasaur,1,318,Gen 1,standard,49,attack
2,bulbasaur,1,318,Gen 1,standard,49,defense
3,bulbasaur,1,318,Gen 1,standard,65,special-attack
4,bulbasaur,1,318,Gen 1,standard,65,special-defense
...,...,...,...,...,...,...,...
6145,baxcalibur,998,600,Gen 9,standard,145,attack
6146,baxcalibur,998,600,Gen 9,standard,92,defense
6147,baxcalibur,998,600,Gen 9,standard,75,special-attack
6148,baxcalibur,998,600,Gen 9,standard,86,special-defense


In [ ]:
dfs = (
    merge_df
    .groupby(['stat_name','gen_num'], as_index = False)
    ['base_stat'].mean()
)

,stat_name,gen_num,base_stat
0,attack,Gen 1,72.913907
1,attack,Gen 2,68.260000
2,attack,Gen 3,73.111111
3,attack,Gen 4,80.214953
4,attack,Gen 5,81.032051
5,attack,Gen 6,72.500000
6,attack,Gen 7,84.772727
7,attack,Gen 8,82.906250
8,attack,Gen 9,82.425000
9,defense,Gen 1,68.225166


In [40]:
base_lg = px.line(
    dfs,
    x = 'gen_num',
    y = 'base_stat',
    facet_col = 'stat_name',
    labels = {
        'gen_num': 'Generation',
        'base_stat': 'Stat Number',
        'stat_name': 'Stat Name'
    },
    title = 'Out of every base stat, the one that has seen the highest increase across Generations is Attack, while Special Defense has not seen almost any sustained increase',
    subtitle = 'Average Pokemon base stats across Generation',
    width = 2000,
    markers = True
)

base_lg.update_yaxes(rangemode='tozero')

base_lg.show()

A general increase is seen in all types of stats (except for special-defense, where we kind of go up and down around a median value, but we still end Gen 9 higher than Gen 1). Still, attack has seen the highest general increase across all the stats, which is likely what is driving the higher stat spreads.

I decided to facet the line graphs by base stat rather than color-coding them because I knew the color-coding was going to cause an issue when I added in the Pokemon classifications, and because I figured that some of the average stats across Generations would be similar to each other, so some of the data points would be obscured. Having six graphs next to each other also makes it easier to compare the stats at a glance.

In [ ]:
dfsc = (
    merge_df
    .groupby(['stat_name','classification','gen_num'], as_index = False)
    ['base_stat'].mean()
)

,stat_name,classification,gen_num,base_stat
0,attack,legendary,Gen 1,96.250000
1,attack,legendary,Gen 2,99.000000
2,attack,legendary,Gen 3,99.375000
3,attack,legendary,Gen 4,107.222222
4,attack,legendary,Gen 5,118.222222
...,...,...,...,...
157,speed,standard,Gen 5,63.426573
158,speed,standard,Gen 6,64.333333
159,speed,standard,Gen 7,62.027778
160,speed,standard,Gen 8,63.250000


In [44]:
base_lg_c = px.line(
    dfsc,
    x = 'gen_num',
    y = 'base_stat',
    color = 'classification',
    facet_col = 'stat_name',
    labels = {
        'gen_num': 'Generation',
        'base_stat': 'Stat Number',
        'stat_name': 'Stat Name',
        'classification': 'Pokemon Classification'
    },
    title = 'For Standard Pokemon, a general increase is seen in the stats across Generations, except for Speed. Legendaries and Mythicals do not seem to have a discernable pattern.',
    subtitle = 'Average base stats for Pokemon across Generations, split up by Pokemon Classification.',
    width = 2000,
    markers = True
)

base_lg_c.update_yaxes(rangemode='tozero')

base_lg_c.show()

When it comes to the standards, we do still see the general increase over time in all stats (except for, once again, special defense and speed, hp doesn't show much increase) but even though not all stats have a general upward increase, they all hover higher than Generation 1 after a certain point (most often Gens 4 and 5, except for speed when we only get more speed in Gen 9). Legendaries and Mythicals are all over the place, however, and this is likely due to the fact that they all concentrate around the same power levels (according to the histograms and the line graphs). Once again, at least for Standards, we do see that attack remains overall the highest stat. 

Also shows that, again, Legendaries and Mythicals are generally stronger than standards.

## Histograms showing the distribution of base stats of Pokemon across the Generations

### Not separated by Pokemon Classification

In [ ]:
def create_stats_histogram(dataframe, stat_column, stat, x_val, color_val, facet_val, wdth, hgt):
##Creates a histogram for showing the distribution of base stats by taking in a dataframe, the desired stat to look at, ##
## what column should be used for the x-axis, the color, and the faceting, and the width and height of the graphs.##
    stats = dataframe[dataframe[stat_column] == stat]
    stat_hist = px.histogram(
        stats,
        x = x_val,
        color = color_val,
        facet_row = facet_val,
        width = wdth,
        height = hgt
    )

    return stat_hist

In [49]:
stat_hst_att = create_stats_histogram(merge_df, 'stat_name', 'attack', 'base_stat','gen_num','gen_num',1000,1000)

stat_hst_att.show()

In [51]:
stat_hst_hp = create_stats_histogram(merge_df, 'stat_name', 'hp', 'base_stat','gen_num','gen_num',1000,1000)

stat_hst_hp.show()

In [53]:
stat_hst_def = create_stats_histogram(merge_df, 'stat_name', 'defense', 'base_stat','gen_num','gen_num',1000,1000)

stat_hst_def.show()

In [54]:
stat_hst_spatt = create_stats_histogram(merge_df, 'stat_name', 'special-attack', 'base_stat','gen_num','gen_num',1000,1000)

stat_hst_spatt.show()

In [55]:
stat_hst_spedef = create_stats_histogram(merge_df, 'stat_name', 'special-defense', 'base_stat','gen_num','gen_num',1000,1000)

stat_hst_spedef.show()

In [56]:
stat_hst_spd = create_stats_histogram(merge_df, 'stat_name', 'speed', 'base_stat','gen_num','gen_num',1000,1000)

stat_hst_spd.show()

These histograms are messy (no altering of the bins, no title or subtitle), but just from a first glance, the rightward shift of many of the histograms over the Generations for each of the stats is obvious, and what I expected to see based on the line graph of averages. The histograms that experience the most obvious rightward shift are the attack stat histograms, once again in line with the line graph of averages. The special defense and speed stat histograms experience the least obvious rightward shifts, in line with the line graph of averages, but the rightward shift is definitely still there.

Now, to see if this pattern of lining up with the line graph continues when we separate the Pokemon out by classification.

### Separated by Pokemon Classification

In [57]:
def create_stats_histogram_class(dataframe, stat_column, stat, class_column, mon_class, x_val, color_val, facet_val, wdth, hgt):
##Creates a histogram for showing the distribution of base stats by taking in a dataframe, the desired stat to look at, ##
## what column should be used for the x-axis, the color, and the faceting, and the width and height of the graphs.##
## Also takes the classification as an additional data filter. ##
    stats = dataframe[(dataframe[stat_column] == stat) & (dataframe[class_column] == mon_class)]
    stat_hist = px.histogram(
        stats,
        x = x_val,
        color = color_val,
        facet_row = facet_val,
        width = wdth,
        height = hgt
    )

    return stat_hist

In [59]:
stat_hst_att_std = create_stats_histogram_class(merge_df, 'stat_name', 'attack', 'classification', 'standard', 'base_stat', 'gen_num', 'gen_num',1000,1000)

stat_hst_att_std.show()

In [60]:
stat_hst_hp_std = create_stats_histogram_class(merge_df, 'stat_name', 'hp', 'classification', 'standard', 'base_stat', 'gen_num', 'gen_num',1000,1000)

stat_hst_hp_std.show()

In [61]:
stat_hst_def_std = create_stats_histogram_class(merge_df, 'stat_name', 'defense', 'classification', 'standard', 'base_stat', 'gen_num', 'gen_num',1000,1000)

stat_hst_def_std.show()

In [62]:
stat_hst_spatt_std = create_stats_histogram_class(merge_df, 'stat_name', 'special-attack', 'classification', 'standard', 'base_stat', 'gen_num', 'gen_num',1000,1000)

stat_hst_spatt_std.show()

In [63]:
stat_hst_spedef_std = create_stats_histogram_class(merge_df, 'stat_name', 'special-defense', 'classification', 'standard', 'base_stat', 'gen_num', 'gen_num',1000,1000)

stat_hst_spedef_std.show()

In [64]:
stat_hst_spd_std = create_stats_histogram_class(merge_df, 'stat_name', 'speed', 'classification', 'standard', 'base_stat', 'gen_num', 'gen_num',1000,1000)

stat_hst_spd_std.show()

In [65]:
stat_hst_att_leg = create_stats_histogram_class(merge_df, 'stat_name', 'attack', 'classification', 'legendary', 'base_stat', 'gen_num', 'gen_num',1000,1000)

stat_hst_att_leg.show()

In [66]:
stat_hst_hp_leg = create_stats_histogram_class(merge_df, 'stat_name', 'hp', 'classification', 'legendary', 'base_stat', 'gen_num', 'gen_num',1000,1000)

stat_hst_hp_leg.show()

In [67]:
stat_hst_def_leg = create_stats_histogram_class(merge_df, 'stat_name', 'defense', 'classification', 'legendary', 'base_stat', 'gen_num', 'gen_num',1000,1000)

stat_hst_def_leg.show()

In [68]:
stat_hst_spatt_leg = create_stats_histogram_class(merge_df, 'stat_name', 'special-attack', 'classification', 'legendary', 'base_stat', 'gen_num', 'gen_num',1000,1000)

stat_hst_spatt_leg.show()

In [69]:
stat_hst_spedef_leg = create_stats_histogram_class(merge_df, 'stat_name', 'special-defense', 'classification', 'legendary', 'base_stat', 'gen_num', 'gen_num',1000,1000)

stat_hst_spedef_leg.show()

In [70]:
stat_hst_spd_leg = create_stats_histogram_class(merge_df, 'stat_name', 'speed', 'classification', 'legendary', 'base_stat', 'gen_num', 'gen_num',1000,1000)

stat_hst_spd_leg.show()

In [71]:
stat_hst_att_myth = create_stats_histogram_class(merge_df, 'stat_name', 'attack', 'classification', 'mythical', 'base_stat', 'gen_num', 'gen_num',1000,1000)

stat_hst_att_myth.show()

In [72]:
stat_hst_hp_myth = create_stats_histogram_class(merge_df, 'stat_name', 'hp', 'classification', 'mythical', 'base_stat', 'gen_num', 'gen_num',1000,1000)

stat_hst_hp_myth.show()

In [73]:
stat_hst_def_myth = create_stats_histogram_class(merge_df, 'stat_name', 'defense', 'classification', 'mythical', 'base_stat', 'gen_num', 'gen_num',1000,1000)

stat_hst_def_myth.show()

In [74]:
stat_hst_spatt_myth = create_stats_histogram_class(merge_df, 'stat_name', 'special-attack', 'classification', 'mythical', 'base_stat', 'gen_num', 'gen_num',1000,1000)

stat_hst_spatt_myth.show()

In [75]:
stat_hst_spedef_myth = create_stats_histogram_class(merge_df, 'stat_name', 'special-defense', 'classification', 'mythical', 'base_stat', 'gen_num', 'gen_num',1000,1000)

stat_hst_spedef_myth.show()

In [76]:
stat_hst_spd_myth = create_stats_histogram_class(merge_df, 'stat_name', 'speed', 'classification', 'mythical', 'base_stat', 'gen_num', 'gen_num',1000,1000)

stat_hst_spd_myth.show()

The histograms for each of the classifications (Standard, Legendary, and Mythical) match up pretty well to the line graph of averages. I feel that the rightward shift for the Standard Pokemon histograms for all stats, except for speed, exhibit an even stronger rightward shift than the histograms for all Pokemon.The speed histograms hardly exhibit a rightward shift at all. As for the Legendary and Mythical Pokemon, the histograms, much like the line graphs, are all over the place and are almost impossible to glean a pattern from. However, looking at this histograms made something very obvious to me, even though this is an observation I have been making throughout the whole notebook: there have been more Legendaries and Mythicals added in each Generation after Generation 1. I also know that there have been less Pokemon added in later Generations (with the exception of Gen 9, which has added the third most Pokemon out of all Generations). 

If more Legendaries and Mythicals are being added per Generation with less Pokemon overall, that means a more powerful class of Pokemon (as shown by both the average line graphs for BSTs and base stats) is becoming increasingly common, which is a sign of power creep. 

Overall, the histograms, at least for all Pokemon and Standard Pokemon, do provide further evidence for power creep: base stats, overall, have been getting higher with each Generation, which is in line with earlier observations that that average BST for all Pokemon and Standard Pokemon have been increasing in later generations.

## Proportions of stronger Pokemon in each Generation

### Proportion of Legendary, Mythical, and Standard Pokemon per Generation

After making the obervation in the base stat histograms that there have been more Legendaries and Mythicals added in later Generations, I wanted to see if the proportion of Legendaries and Mythicals added in each Generation has increased since Generation one. I also wanted to see if the proportion of more powerful Pokemon added in each Generation has increased, as well, in terms of just Standard Pokemon and Pokemon as a whole.

#### What do you define as a powerful Pokemon?

I am defining a powerful Pokemon as a Pokemon with at least 500 BST. I have arrived at this benchmark through a number of sources. The most relevant:
- The average BST of all Pokemon in Generation 1 is around 400 BST. Legendaries and Mythicals, a more powerful class of Pokemon, start at around 600 BST. Take the average, you arrive at about 500 BST. Because I want to compare everything to Generation 1, I am keeping this as my benchmark for all Generations, even when the average BST has increased.
- Looking at the [upper echelon of BSTs for each type, excluding Legendaries and Mythicals](https://pokemondb.net/pokebase/355610/what-pokemon-have-the-best-stat-totals-of-each-type) for Generation 8, the most powerful Pokemon tend to start at the 500 BST range. This is a Generation behind the most recent, but these totals should hold for Generation 9
- A full list of other sources consulted when determining this threshold can be found in the README. 

#### Why not calculate a moving average?

A moving average makes it hard to compare proportions across Generations. Calculating one baseline makes comparisons fairer.

Calculating a moving average for what makes a powerful Pokemon, however, could be a good way to measure power creep. If the benchmark for what makes a powerful Pokemon changes from Generation to Generation, then power creep is occurring.

In [ ]:
df_base

,is_default,pokemon_name,pokemon_url,id,name,is_baby,is_legendary,is_mythical,forms_switchable,has_gender_differences,gen,gen_url,BST,gen_num,Classification
0,True,bulbasaur,https://pokeapi.co/api/v2/pokemon/1/,1,bulbasaur,False,False,False,False,False,generation-i,https://pokeapi.co/api/v2/generation/1/,318,Gen 1,Standard
1,True,charmander,https://pokeapi.co/api/v2/pokemon/4/,4,charmander,False,False,False,False,False,generation-i,https://pokeapi.co/api/v2/generation/1/,309,Gen 1,Standard
2,True,squirtle,https://pokeapi.co/api/v2/pokemon/7/,7,squirtle,False,False,False,False,False,generation-i,https://pokeapi.co/api/v2/generation/1/,314,Gen 1,Standard
3,True,caterpie,https://pokeapi.co/api/v2/pokemon/10/,10,caterpie,False,False,False,False,False,generation-i,https://pokeapi.co/api/v2/generation/1/,195,Gen 1,Standard
4,True,weedle,https://pokeapi.co/api/v2/pokemon/13/,13,weedle,False,False,False,False,False,generation-i,https://pokeapi.co/api/v2/generation/1/,195,Gen 1,Standard
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1020,True,gholdengo,https://pokeapi.co/api/v2/pokemon/1000/,1000,gholdengo,False,False,False,False,False,generation-ix,https://pokeapi.co/api/v2/generation/9/,550,Gen 9,Standard
1021,True,dipplin,https://pokeapi.co/api/v2/pokemon/1011/,1011,dipplin,False,False,False,False,False,generation-ix,https://pokeapi.co/api/v2/generation/9/,485,Gen 9,Standard
1022,True,sinistcha,https://pokeapi.co/api/v2/pokemon/1013/,1013,sinistcha,False,False,False,False,False,generation-ix,https://pokeapi.co/api/v2/generation/9/,508,Gen 9,Standard
1023,True,archaludon,https://pokeapi.co/api/v2/pokemon/1018/,1018,archaludon,False,False,False,False,False,generation-ix,https://pokeapi.co/api/v2/generation/9/,600,Gen 9,Standard


#### Creating a dataframe to be used in the analysis

I don't have the numbers yet to perform this analysis. I need to get the count of Pokemon in each Generation, as well as the count of all Standard, Legenary, and Mythical Pokemon in each Generation. Dividing the count of each class of Pokemon by the count of Pokemon in each Generation will give me the proportions I need. I also want a count of all Standard Pokemon with BSTs over and under 500 BST, and a count of all Pokemon with BSTs over and under 500 BST. Combining all of these separate measures into one dataframe will provide the basis for my stacked bar charts of proportions.

I will be using the df_base dataframe for this analysis, since the merge_df has six rows for each Pokemon, which will skew my data.

In [90]:
def over_500(row):
##Checks if a Pokemon has a BST over 500. If it does, assign Yes to the row. If it doesn't, assign No.##
    if row['BST'] >= 500:
        return 'Yes'
    else:
        return 'No'

In [91]:
df_base['over_500'] = df_base.apply(lambda row: over_500(row), axis = 1)

mon_count = (
    df_base.groupby(['gen_num'], as_index = False)['pokemon_name'].count().rename(columns = {'pokemon_name':'pokemon_in_gen'})
)

standard_mon = (
    df_base[
        df_base['classification'] == 'standard'
    ]
    .groupby(['gen_num'], as_index = False)
    ['pokemon_name'].count()
    .drop(columns = ['gen_num'])
    .rename(columns = {'pokemon_name':'standard_pokemon_in_gen'})
)

legendary_mon = (
    df_base[
        df_base['classification'] == 'legendary'
    ]
    .groupby(['gen_num'], as_index = False)
    ['pokemon_name'].count()
    .drop(columns = ['gen_num'])
    .rename(columns = {'pokemon_name':'legendary_pokemon_in_gen'})
)

mythical_mon = (
    df_base[
        df_base['classification'] == 'mythical'
    ]
    .groupby(['gen_num'], as_index = False)
    ['pokemon_name'].count()
    .drop(columns = ['gen_num'])
    .rename(columns = {'pokemon_name':'mythical_pokemon_in_gen'})
)

mon_over_500 = (
    df_base[
    df_base['over_500'] == 'Yes'
    ]
    .groupby(['gen_num'], as_index = False)
    ['pokemon_name'].count()
    .drop(columns = ['gen_num'])
    .rename(columns = {'pokemon_name':'pokemon_over_500_bst'})
)

standard_over_500 = (
    df_base[
        (df_base['classification'] == 'standard')
        &
        (df_base['over_500'] == 'Yes')
    ]
    .groupby(['gen_num'], as_index = False)
    ['pokemon_name'].count()
    .drop(columns = ['gen_num'])
    .rename(columns = {'pokemon_name':'standard_pokemon_over_500'})
)

prop_df = pd.concat([mon_count, standard_mon, legendary_mon, mythical_mon, mon_over_500, standard_over_500], axis = 1)

prop_df

,gen_num,pokemon_in_gen,standard_pokemon_in_gen,legendary_pokemon_in_gen,mythical_pokemon_in_gen,pokemon_over_500_bst,standard_pokemon_over_500
0,Gen 1,151,146,4,1,33,28
1,Gen 2,100,94,5,1,25,19
2,Gen 3,135,125,8,2,23,13
3,Gen 4,107,93,9,5,41,28
4,Gen 5,156,143,9,4,37,24
5,Gen 6,72,66,3,3,23,17
6,Gen 7,88,72,11,5,33,20
7,Gen 8,96,84,11,1,39,28
8,Gen 9,120,108,11,1,56,45


In [92]:
prop_df = prop_df.assign(prop_over_500 = prop_df['pokemon_over_500_bst']/prop_df['pokemon_in_gen'])
prop_df = prop_df.assign(prop_standard = prop_df['standard_pokemon_in_gen']/prop_df['pokemon_in_gen'])
prop_df = prop_df.assign(prop_legendary = prop_df['legendary_pokemon_in_gen']/prop_df['pokemon_in_gen'])
prop_df = prop_df.assign(prop_mythical = prop_df['mythical_pokemon_in_gen']/prop_df['pokemon_in_gen'])
prop_df = prop_df.assign(prop_standard_500 = prop_df['standard_pokemon_over_500']/prop_df['pokemon_in_gen'])
prop_df = prop_df.assign(prop_standard_under_500 = 1 - prop_df['prop_standard_500'])
prop_df = prop_df.assign(prop_under_500 = 1 - prop_df['prop_over_500'])

prop_df

,gen_num,pokemon_in_gen,standard_pokemon_in_gen,legendary_pokemon_in_gen,mythical_pokemon_in_gen,pokemon_over_500_bst,standard_pokemon_over_500,prop_over_500,prop_standard,prop_legendary,prop_mythical,prop_standard_500,prop_standard_under_500,prop_under_500
0,Gen 1,151,146,4,1,33,28,0.218543,0.966887,0.026490,0.006623,0.185430,0.814570,0.781457
1,Gen 2,100,94,5,1,25,19,0.250000,0.940000,0.050000,0.010000,0.190000,0.810000,0.750000
2,Gen 3,135,125,8,2,23,13,0.170370,0.925926,0.059259,0.014815,0.096296,0.903704,0.829630
3,Gen 4,107,93,9,5,41,28,0.383178,0.869159,0.084112,0.046729,0.261682,0.738318,0.616822
4,Gen 5,156,143,9,4,37,24,0.237179,0.916667,0.057692,0.025641,0.153846,0.846154,0.762821
5,Gen 6,72,66,3,3,23,17,0.319444,0.916667,0.041667,0.041667,0.236111,0.763889,0.680556
6,Gen 7,88,72,11,5,33,20,0.375000,0.818182,0.125000,0.056818,0.227273,0.772727,0.625000
7,Gen 8,96,84,11,1,39,28,0.406250,0.875000,0.114583,0.010417,0.291667,0.708333,0.593750
8,Gen 9,120,108,11,1,56,45,0.466667,0.900000,0.091667,0.008333,0.375000,0.625000,0.533333


In [94]:
leg_and_myth_bp = px.bar(
    prop_df,
    x = 'gen_num',
    y = ['prop_standard','prop_legendary','prop_mythical'],
    title = 'The proportion of Legendaries and Mythicals compared to Standard Pokemon has been increasing over Generations',
    subtitle = 'Proportion of Pokemon added in each class over Generations 1-9 of Pokemon',
    labels = {
        'gen_num':'Generation',
        'value':'Proportion out of 1',
        'variable': 'Proportion of Pokemon Types'
    },
    text_auto = True,
    height = 1000,
    width = 1500,
    barmode = 'stack'
)

leg_and_myth_bp.show()

The proportion of Legendary Pokemon added in each Generation has not increased consistently over time, but it is definitely higher than it was in Generation 1, reaching its peak in Generation 7 with 1/8 of the Pokemon added in that Generation being Legendary. Mythical Pokemon have returned to about Generation 1 proportions in Generation 8 and 9, but from Generations 3 through 7, the proportion of Mythical Pokemon is higher than in Generation 1. This provides strong evidence for my earlier claim that Mythicals and Legendaries, a more powerful class of Pokemon, are becoming more common in newer Generations.

### Proportion of Standard Pokemon over 500 BST per Generation

In [95]:
std_500_bp = px.bar(
    prop_df,
    x = 'gen_num',
    y = ['prop_standard_under_500', 'prop_standard_500'],
    title = 'Though Pokemon per Generation is generally getting smaller, the proportion of Pokemon over 500 BST is getting larger',
    subtitle = 'Proportion of Pokemon over or under the threshold of 500 BST over Generations 1-9',
    labels = {
        'gen_num':'Generation',
        'value':'Proportion out of 1',
        'variable': 'Proportion of standard Pokemon over/under a certain stat threshold'
    },
    text_auto = True,
    height = 1000,
    width = 2000
)

std_500_bp.show()

This stacked bar chart provides strong evidence of power creep, as Standard Pokemon over 500 BST, at least since Generation 5, has been increasing by Generation, reaching its peak in the most recent Generation. Finding the proportion of Standard Pokemon over and under 500 BST was a worthwhile endeavor.

### Proportion of all Pokemon over 500 BST per Generation

In [99]:
over_500_bp = px.bar(
    prop_df,
    x = 'gen_num',
    y = ['prop_under_500', 'prop_over_500'],
    title = 'After Generation 5, Pokemon with over 500 BST have been steadily increasing for each generation.',
    subtitle = 'Nearly half of all 120 Pokemon in Gen 9 have BST over 500, compared to around 22 percent in our Gen 1 baseline. Proportion of Pokemon over or under the threshold of 500 BST, Generations 1-9',
    labels = {
        'gen_num':'Generation',
        'value':'Proportion out of 1',
        'variable': 'Proportion of all Pokemon over/under a certain stat threshold'
    },
    text_auto = True,
    height = 1000,
    width = 2000
)

over_500_bp.show()

If Standard Pokemon over 500 BST has been increasing over Generations, and more Legendary and Mythical Pokemon have been added over Generations, then it makes sense that including all Pokemon in a stacked bar chart would increase the overall proportions of Pokemon over 500 BST per Generation, making this bar graph a little redundant compared to the earlier two. However, seeing that almost half of all Pokemon added in Generation 9 were over 500 BST is a staggering figure and provides strong evidence of power creep.

## Limitiations

Though strong evidence of power creep has been provided in my analyses, there are some limitations to consider. First, my filtering of the Pokemon was a bit indiscriminate, with me filtering out Pokemon I would have liked to keep (ex: the regional forms) and Pokemon being kept in that I would have liked to remove (ex: Terapagos Normal Form). Though the filtering was about 90% accurate, if the dataset was altered in the way that I would have originally liked it to be, the data and the graphs could look very different. Second, this conclusion of power creep only applies to the Pokemon and their stats, not to any other aspect of Pokemon, so it is hard to make conclusions about power creep as a whole. There is evidence that Pokemon overall are getting stronger with higher BSTs over time, but what about abilities? Moves? Items? Are those getting stronger, edging out earlier abilities, moves, and items, or are they staying the same and it is only the Pokemon themselves being affected. Additionally, no Generational gimmick was considered in this analysis, but the introduction of gimmicks overall could also be another avenue for power creep. Finally, only current BSTs and base stats have been considered in this analysis, meaning Pokemon that have had their BSTs or base stats altered at any point in the past are working with updated data. A better analysis would have used the original BSTs and base stats from the Generation that the Pokemon was introduced in so as to get a more accurate picture of how BSTs and stats changed over time. This has led to some skew in the data, but I do not think it is substantial enough to completely discredit my findings.

## Conclusion

Has Pokemon experienced power creep over its thirty years of existence? I believe that yes, it has. Not only are average BSTs, and thus average base stats, increasing with each Generation, but Legendaries and Mythicals are becoming more common with each Generation, as are Pokemon over 500 BST. To what extent has power creep been experienced? Based on my evidence, it is hard to make an exact judgement on the extent of power creep experienced, but I do not believe it is to an extreme extent. Pokemon are stronger now than they were in the past, but not drastically. Additionally, I can only make claims of power creep on Pokemon stats themselves; I cannot make claims about the game as a whole. So, when it comes to stats, Pokemon has experienced a moderate amount of power creep.

Future analyses could alter this data in ways mentioned in the limitations section, or could explore other facets of the game of Pokemon to get a more complete picture of power creep.